In [0]:
from pyspark.sql.functions import *

In [0]:
# Lire la table transactions Bronze
df_transactions = spark.table("`E-commerce`.bronze.transactions")

In [0]:
# Afficher les colonnes et les types
df_transactions.printSchema()

In [0]:
# Supprimer les lignes identiques
transactions_silver = df_transactions.dropDuplicates()

In [0]:
# Garder une seule ligne par transaction_id
transactions_silver = transactions_silver.dropDuplicates(["transaction_id"])

In [0]:
# Supprimer les transactions sans transaction_id
transactions_silver = transactions_silver.filter(
    col("transaction_id").isNotNull()
)

In [0]:
# Standardiser les moyens de paiement
transactions_silver = transactions_silver.withColumn(
    "payment_method",
    lower(trim(col("payment_method")))
)

In [0]:
# Remplacer les quantites negatives par NULL
from pyspark.sql.functions import when

transactions_silver = transactions_silver.withColumn(
    "quantity",
    when(col("quantity") < 0, None)
    .otherwise(col("quantity"))
)

In [0]:
# Remplacer les prix negatifs par NULL
transactions_silver = transactions_silver.withColumn(
    "unit_price",
    when(col("unit_price") < 0, None)
    .otherwise(col("unit_price"))
)

In [0]:
# Trouver les montants incoherents
invalid_amounts = transactions_silver.filter(
    (col("quantity").isNotNull()) &
    (col("unit_price").isNotNull()) &
    (col("total_amount") != col("quantity") * col("unit_price"))
)

display(invalid_amounts)

In [0]:
# Trouver les transactions avec un client inexistant
invalid_transaction_customers = (
    transactions_silver
    .join(
        spark.table("`E-commerce`.silver.customers").select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

display(invalid_transaction_customers)

In [0]:
# Trouver les transactions avec un produit inexistant
invalid_transaction_products = (
    transactions_silver
    .join(
        spark.table("`E-commerce`.silver.products").select("product_id"),
        "product_id",
        "left_anti"
    )
)

display(invalid_transaction_products)

In [0]:
# Recalculer le montant total
transactions_silver = transactions_silver.withColumn(
    "total_amount",
    when(
        col("quantity").isNotNull() & col("unit_price").isNotNull(),
        col("quantity") * col("unit_price")
    ).otherwise(col("total_amount"))
)

In [0]:
# Garder les transactions avec un client valide
transactions_silver = transactions_silver.join(
    spark.table("`E-commerce`.silver.customers").select("customer_id"),
    "customer_id",
    "inner"
)

In [0]:
# Comparer Bronze et Silver
print("Bronze :", df_transactions.count())
print("Silver :", transactions_silver.count())

In [0]:
# Enregistrer transactions dans Silver
transactions_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.silver.transactions")